# 06주차: PyTorch Fashion-MNIST 이미지 분류 기초

## 학습 목표
Fashion-MNIST 흑백 의류 이미지를 텐서로 읽고, 합성곱 신경망으로 열 가지 클래스를 분류합니다. 학습·검증·테스트 데이터의 역할을 구분하고, 로짓·softmax 확률·정확도를 관찰합니다. 학습 전의 무작위 예측과 학습 후 예측을 같은 테스트 이미지에서 비교하며 신경망이 데이터를 통해 바뀌는 과정을 이해합니다.

Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. GPU가 없어도 CPU에서 실행되지만 학습이 오래 걸릴 수 있습니다. Fashion-MNIST는 자동으로 내려받으므로 Google Drive 연결, Drive 마운트, 파일 업로드는 필요하지 않습니다.

## 관찰 질문
- 학습 전 모델의 확률 합은 왜 1이고, 예측은 왜 정답과 자주 다를까요?
- 학습 데이터와 검증 데이터, 테스트 데이터를 분리하는 이유는 무엇일까요?
- 오분류된 두 의류 클래스는 이미지의 어떤 모양 때문에 헷갈릴까요?


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

transform = transforms.ToTensor()
full_train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_train_dataset, [54000, 6000], generator=split_generator)
batch_size = 128
loader_options = {"batch_size": batch_size, "num_workers": 2, "pin_memory": torch.cuda.is_available()}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print("데이터 수:", len(train_dataset), len(val_dataset), len(test_dataset))


## 데이터 시각화와 모델
`ToTensor`는 28×28 회색조 이미지를 0에서 1 사이의 한 채널 텐서로 바꿉니다. 아래 2×5 표본에서 라벨과 이미지를 함께 확인합니다. `SmallCNN`은 두 개의 합성곱 층과 풀링 층으로 특징을 추출하고, 마지막 분류기가 열 개 클래스의 점수를 만듭니다. 입력과 출력 텐서의 shape도 확인해 보세요.


In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
sample_images, sample_labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for image, label, axis in zip(sample_images[:10], sample_labels[:10], axes.flat):
    axis.imshow(image.squeeze(0), cmap="gray")
    axis.set_title(class_names[label.item()])
    axis.axis("off")
plt.tight_layout()
plt.show()
print("입력 텐서 shape:", sample_images.shape)

class SmallCNN(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

with torch.no_grad():
    shape_logits = SmallCNN(in_channels=1).to(device)(sample_images.to(device))
print("출력 텐서 shape:", shape_logits.shape)


## 학습 전 모델 출력 실험
학습되지 않은 모델은 아직 정답을 본 적이 없습니다. 로짓은 클래스별 점수이고, softmax는 이를 확률로 바꾸어 각 이미지의 확률 합을 1로 만듭니다. 전체 테스트셋 초기 정확도는 정확히 10%라고 단정하지 않고 우연 수준에 가까운지 관찰합니다. 같은 `images`를 나중에 학습 후 모델에도 넣어 비교합니다.


In [ ]:
def measure_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in loader:
            logits = model(batch_images.to(device))
            correct += (logits.argmax(dim=1).cpu() == batch_labels).sum().item()
            total += batch_labels.size(0)
    return correct / total

untrained_model = SmallCNN(in_channels=1).to(device)
images, labels = next(iter(test_loader))
with torch.no_grad():
    initial_logits = untrained_model(images.to(device))
    initial_probabilities = torch.softmax(initial_logits, dim=1)
initial_predictions = initial_probabilities.argmax(dim=1).cpu()
initial_accuracy = measure_accuracy(untrained_model, test_loader)
print("로짓 예시:", initial_logits[0].cpu())
print("확률 합:", initial_probabilities[0].sum().item())
print("초기 예측과 정답:", class_names[initial_predictions[0].item()], class_names[labels[0].item()])
print(f"학습 전 테스트 정확도: {initial_accuracy:.2%}; 우연 수준에 가까운지 해석하세요.")


## 학습·평가 함수와 5에포크 학습
`train_one_epoch`는 한 번의 학습 데이터 순회에서 가중치를 갱신합니다. `evaluate`는 가중치를 바꾸지 않고 손실과 정확도를 계산합니다. `fit`은 매 에포크의 학습·검증 기록을 쌓고, `plot_history`는 변화 추이를 그립니다. CrossEntropyLoss와 Adam 최적화기를 사용합니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for batch_images, batch_labels in loader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            logits = model(batch_images)
            loss_sum += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
        for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%")
    return history

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].legend()
    plt.show()

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

model = SmallCNN(in_channels=1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("학습 가능한 파라미터 수:", count_parameters(model))
history = fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=5)
plot_history(history, "SmallCNN")
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
print(f"테스트 손실: {test_loss:.4f}, 테스트 정확도: {test_accuracy:.2f}%")


## 학습 전후 비교와 학생 활동
동일한 테스트 이미지의 초기 예측과 학습 후 예측을 비교하고, 틀린 이미지는 최대 10개만 모아 살펴봅니다. 오분류는 실패 목록이 아니라 다음 실험의 단서입니다. 비슷한 옷의 윤곽, 낮은 해상도, 제한된 모델 용량 중 무엇이 원인인지 추론해 보세요.

### 학생 활동
1. 첫 합성곱 층 뉴런 수 16을 8, 32, 64로 바꾸고 파라미터 수, 시간, 정확도를 비교하세요.
2. 에포크를 1, 5, 10으로 바꾸고 학습·검증 곡선에서 과소적합 또는 과적합의 신호를 설명하세요.
3. 오분류에서 자주 함께 나타나는 두 클래스를 골라 사람이 구분하는 단서를 적어 보세요.

### 7주차 연결 질문
다음 주에는 세 채널 컬러 CIFAR-10에 같은 `SmallCNN` 구조를 적용합니다. `in_channels`만 3으로 바꾸면 어떤 층의 입력이 달라질까요? 데이터 증강과 더 깊은 모델은 오늘의 기본 모델 한계를 어떻게 보완할까요?


In [ ]:
model.eval()
with torch.no_grad():
    trained_logits = model(images.to(device))
trained_predictions = trained_logits.argmax(dim=1).cpu()
for index in range(10):
    print(f"{index}: 초기={class_names[initial_predictions[index].item()]}, 학습 후={class_names[trained_predictions[index].item()]}, 정답={class_names[labels[index].item()]}")

mistake_images, mistake_predictions, mistake_labels = [], [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_predictions = model(batch_images.to(device)).argmax(dim=1).cpu()
        for image, prediction, label in zip(batch_images[batch_predictions != batch_labels], batch_predictions[batch_predictions != batch_labels], batch_labels[batch_predictions != batch_labels]):
            mistake_images.append(image)
            mistake_predictions.append(prediction)
            mistake_labels.append(label)
            if len(mistake_images) == 10:
                break
        if len(mistake_images) == 10:
            break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, image, prediction, label in zip(axes.flat, mistake_images, mistake_predictions, mistake_labels):
    axis.imshow(image.squeeze(0), cmap="gray")
    axis.set_title(f"Pred: {class_names[prediction.item()]}\nActual: {class_names[label.item()]}")
    axis.axis("off")
for axis in axes.flat[len(mistake_images):]:
    axis.axis("off")
plt.tight_layout()
plt.show()
